In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import math
from scipy.stats import chi2

# Data

In [ ]:
# Membuat data
data_kedatangan = {"Hari": ["Hari 1", "Hari 2", "Hari 3"],
                   "10-11": [180, 159, 160],
                   "11-12": [140, 160, 195],
                   "12-13": [60, 80, 50]}
data_pelayanan = {"Hari": ["Hari 1", "Hari 2", "Hari 3"],
                  "10-11": [50, 59, 60],
                  "11-12": [45, 60, 90],
                  "12-13": [30, 40, 60]}
kedatangan_df = pd.DataFrame(data_kedatangan)
pelayanan_df = pd.DataFrame(data_pelayanan)

print("Data kedatangan:")
display(kedatangan_df)
print("\nData Pelayanan:")
display(pelayanan_df)

Data kedatangan:


,Hari,10-11,11-12,12-13
0,Hari 1,180,140,60
1,Hari 2,159,160,80
2,Hari 3,160,195,50



Data Pelayanan:


,Hari,10-11,11-12,12-13
0,Hari 1,50,45,30
1,Hari 2,59,60,40
2,Hari 3,60,90,60


# (M/M/s)

In [ ]:
total_kedatangan = kedatangan_df[["10-11", "11-12", "12-13"]].to_numpy().sum()
total_pelayanan = pelayanan_df[["10-11", "11-12", "12-13"]].to_numpy().sum()
n_interval = 9

# Menghitung λ dan μ
λ = total_kedatangan / n_interval
μ = total_pelayanan / n_interval

In [ ]:
# fungsi (M/M/s)
def hitung_mms(lambd, mu, s):
    if mu <= 0 or lambd <= 0 or s < 1:
        print("λ dan μ harus > 0, dan s harus >= 1")
        return None

    rho = lambd / (s * mu)
    if rho >= 1:
        print(f"ρ = λ/(s μ) = {rho:.4f}≥ 1 → Sistem tidak stabil (λ harus < sμ)\n")

    r = lambd / mu

    sum_finite = 0.0
    for n in range(s):
        sum_finite += (r**n)/math.factorial(n)
    tail_term = (r**s)/(math.factorial(s)*(1-rho))

    P0 = 1/(sum_finite + tail_term)
    Lq = P0*(r**s)*rho/(math.factorial(s)*(1-rho)**2)
    L = Lq + lambd/mu
    Wq = Lq/lambd
    W = Wq + 1/mu
    Cs = 30000
    Cw = 10000
    service_cost = Cs * s
    waiting_cost = Cw * Lq
    total_cost = service_cost + waiting_cost

    print(f"=== HASIL SIMULASI MODEL ANTRIAN M/M/{s} ===")
    print(f"Jumlah server (s)\t\t: {s}")
    print(f"Laju kedatangan (λ)\t\t: {lambd:.2f} pelanggan/jam")
    print(f"Laju pelayanan (μ)\t\t: {mu:.2f} pelanggan/jam per server")
    print(f"Tingkat kesibukan sistem (ρ)\t: {rho:.4f}")

    # interpretasi rho
    if rho < 0.7:
        print("➡️ Sistem cukup longgar, petugas cenderung sering idle.")
    elif rho < 0.9:
        print("➡️ Sistem dalam kondisi efisien (stabil dan optimal).")
    else:
        print("⚠️ Sistem sangat sibuk, potensi antrean panjang.")

    print("\n=== Parameter Kinerja Sistem ===")
    print(f"Probabilitas tidak ada pelanggan (P0)\t\t: {P0:.4f}")
    print(f"Rata-rata pelanggan dalam antrian (Lq)\t\t: {Lq:.4f}")
    print(f"Rata-rata pelanggan total dalam sistem (L)\t: {L:.4f}")
    print(f"Waktu tunggu rata-rata dalam antrian (Wq)\t: {Wq*60:.2f} menit")
    print(f"Waktu rata-rata dalam sistem (W)\t\t: {W*60:.2f} menit")

    print("\n=== ANALISIS BIAYA ===")
    print(f"Biaya pelayanan per jam : Rp{service_cost:,.0f}")
    print(f"Biaya menunggu per jam  : Rp{waiting_cost:,.0f}")
    print(f"Total biaya per jam     : Rp{total_cost:,.0f}")


    # interpretasi gabungan efisiensi & waktu tunggu
    print("\n📊 Interpretasi:")

    # efisiensi sistem (berdasarkan rho)
    if rho < 0.7:
        print("Sistem tergolong longgar, petugas sering idle dan biaya relatif tinggi.")
    elif rho < 0.9:
        print("Sistem berada pada kondisi efisien (stabil dan seimbang).")
    else:
        print("Sistem overload — beban pelayanan terlalu tinggi, risiko antrean panjang.")

    # waktu tunggu (berdasarkan Wq)
    if Wq * 60 < 5:
        print("⏱️ Waktu tunggu rata-rata sangat baik (<5 menit).")
    elif Wq * 60 < 10:
        print("⌛ Waktu tunggu masih tergolong wajar (5–10 menit).")
    else:
        print("🕒 Waktu tunggu cukup lama, pertimbangkan penambahan server atau peningkatan kecepatan layanan.")

# (M/M/1) kodingan biasa

In [ ]:
# Jika hanya terdapat 1 loket, maka model antriannya adalah (M/M/1)
lambda_val = λ
mu_val = μ
s_val = 1

hasil = hitung_mms(lambda_val, mu_val, s_val)

ρ = λ/(s μ) = 2.3968≥ 1 → Sistem tidak stabil (λ harus < sμ)

=== HASIL SIMULASI MODEL ANTRIAN M/M/1 ===
Jumlah server (s)		: 1
Laju kedatangan (λ)		: 131.56 pelanggan/jam
Laju pelayanan (μ)		: 54.89 pelanggan/jam per server
Tingkat kesibukan sistem (ρ)	: 2.3968
⚠️ Sistem sangat sibuk, potensi antrean panjang.

=== Parameter Kinerja Sistem ===
Probabilitas tidak ada pelanggan (P0)		: -1.3968
Rata-rata pelanggan dalam antrian (Lq)		: -4.1127
Rata-rata pelanggan total dalam sistem (L)	: -1.7159
Waktu tunggu rata-rata dalam antrian (Wq)	: -1.88 menit
Waktu rata-rata dalam sistem (W)		: -0.78 menit

=== ANALISIS BIAYA ===
Biaya pelayanan per jam : Rp30,000
Biaya menunggu per jam  : Rp-41,127
Total biaya per jam     : Rp-11,127

📊 Interpretasi:
Sistem overload — beban pelayanan terlalu tinggi, risiko antrean panjang.
⏱️ Waktu tunggu rata-rata sangat baik (<5 menit).


Karena nilai rata rata waktunya tidak masuk akal, akan dicoba dengan finite horizon

In [ ]:
def hitung_mms_finite_horizon(lambd, mu, s, horizon_hours):
    if mu <= 0 or lambd <= 0 or s <= 0:
        raise ValueError("λ, μ harus > 0 dan s >= 1")

    rho = lambd / (s*mu)

    # Estimasi finite-horizon bila tidak stabil (ρ ≥ 1)
    if rho >= 1:
        # backlog tumbuh sebesar (λ - sμ) per jam selama horizon H
        add_backlog = (lambd - s*mu) * max(horizon_hours, 0.0)
        Lq_est = max(0.0, add_backlog)
        Wq_est = Lq_est / lambd
        W_est  = Wq_est + 1.0/mu
        L_est  = Lq_est + lambd/mu

        Cs, Cw = 30000, 10000
        service_cost = Cs * s
        waiting_cost = Cw * Lq_est
        total_cost   = service_cost + waiting_cost

        return {
            'stable': False,
            'rho': rho, 'P0': 0.0,
            'Lq': Lq_est, 'Wq': Wq_est, 'W': W_est, 'L': L_est,
            'service_cost': service_cost,
            'waiting_cost': waiting_cost,
            'total_cost': total_cost,
            'horizon_hours': horizon_hours}  #penanda ini angka horizon

    # ρ < 1 → rumus M/M/s normal
    r = lambd / mu
    sum_finite = sum((r**n)/math.factorial(n) for n in range(s))
    tail_term  = (r**s)/(math.factorial(s)*(1 - rho))
    P0 = 1.0/(sum_finite + tail_term)

    Lq = P0 * (r**s) * rho / (math.factorial(s) * (1 - rho)**2)
    Wq = Lq / lambd
    W  = Wq + 1.0/mu
    L  = Lq + lambd/mu

    Cs, Cw = 30000, 10000
    service_cost = Cs * s
    waiting_cost = Cw * Lq
    total_cost   = service_cost + waiting_cost

    return {
        'stable': True,
        'rho': rho, 'P0': P0,
        'Lq': Lq, 'Wq': Wq, 'W': W, 'L': L,
        'service_cost': service_cost,
        'waiting_cost': waiting_cost,
        'total_cost': total_cost}
def hasil_mms_finite_horizon(lambd, mu, s, horizon_hours=1):
    hasil = hitung_mms_finite_horizon(lambd, mu, s, horizon_hours)

    print(f"=== HASIL SIMULASI MODEL ANTRIAN M/M/{s} ===")
    print(f"Jumlah server (s): {s_val}")
    print(f"Laju kedatangan (λ): {lambda_val:.2f} pelanggan/jam")
    print(f"Laju pelayanan (μ): {mu_val:.2f} pelanggan/jam per server")
    print(f"Tingkat kesibukan sistem (ρ): {hasil['rho']:.4f}")

    if not hasil['stable']:
        print("⚠️ Sistem tidak stabil (ρ ≥ 1)\n"
              "Angka di bawah ini adalah ESTIMASI untuk horizon "
              f"{hasil['horizon_hours']} jam agar bisa dibandingkan:")
    else:
        print("➡️ Sistem stabil (steady-state).")

    print("\n=== Parameter Kinerja Sistem ===")
    print(f"P0: {hasil['P0']:.4f}")
    print(f"Lq: {hasil['Lq']:.4f}")
    print(f"L : {hasil['L']:.4f}")
    print(f"Wq: {hasil['Wq']*60:.2f} menit")
    print(f"W : {hasil['W']*60:.2f} menit")

    print("\n=== ANALISIS BIAYA ===")
    print(f"💰 Biaya pelayanan per jam : Rp{hasil['service_cost']:,.0f}")
    print(f"💸 Biaya menunggu per jam  : Rp{hasil['waiting_cost']:,.0f}")
    print(f"🧾 Total biaya per jam     : Rp{hasil['total_cost']:,.0f}")

    print("\n📊 Interpretasi:")

    # efisiensi sistem (berdasarkan rho)
    if hasil['rho'] < 0.7:
        print("Sistem tergolong longgar — petugas sering idle, biaya relatif tinggi.")
    elif hasil['rho'] < 0.9:
        print("Sistem berada pada kondisi efisien (stabil dan seimbang).")
    else:
        print("Sistem overload — beban pelayanan terlalu tinggi, risiko antrean panjang.")

    # waktu tunggu (berdasarkan Wq)
    if hasil['Wq'] * 60 < 5:
        print("⏱️ Waktu tunggu rata-rata sangat baik (<5 menit).")
    elif hasil['Wq'] * 60 < 10:
        print("⌛ Waktu tunggu masih tergolong wajar (5–10 menit).")
    else:
        print("🕒 Waktu tunggu cukup lama, "
              "pertimbangkan penambahan server atau peningkatan kecepatan layanan.")

#(M/M/1) finite horizon

In [ ]:
lambda_val = λ
mu_val = μ
s_val = 1
h_hours = 1
hasil = hasil_mms_finite_horizon(lambda_val, mu_val, s_val, h_hours)


=== HASIL SIMULASI MODEL ANTRIAN M/M/1 ===
Jumlah server (s): 1
Laju kedatangan (λ): 131.56 pelanggan/jam
Laju pelayanan (μ): 54.89 pelanggan/jam per server
Tingkat kesibukan sistem (ρ): 2.3968
⚠️ Sistem tidak stabil (ρ ≥ 1)
Angka di bawah ini adalah ESTIMASI untuk horizon 1 jam agar bisa dibandingkan:

=== Parameter Kinerja Sistem ===
P0: 0.0000
Lq: 76.6667
L : 79.0634
Wq: 34.97 menit
W : 36.06 menit

=== ANALISIS BIAYA ===
💰 Biaya pelayanan per jam : Rp30,000
💸 Biaya menunggu per jam  : Rp766,667
🧾 Total biaya per jam     : Rp796,667

📊 Interpretasi:
Sistem overload — beban pelayanan terlalu tinggi, risiko antrean panjang.
🕒 Waktu tunggu cukup lama, pertimbangkan penambahan server atau peningkatan kecepatan layanan.


# (M/M/2) kodingan biasa


In [ ]:
Slambda_val = λ
mu_val = μ
s_val = 2

hasil = hitung_mms(lambda_val, mu_val, s_val)

ρ = λ/(s μ) = 1.1984≥ 1 → Sistem tidak stabil (λ harus < sμ)

=== HASIL SIMULASI MODEL ANTRIAN M/M/2 ===
Jumlah server (s)		: 2
Laju kedatangan (λ)		: 131.56 pelanggan/jam
Laju pelayanan (μ)		: 54.89 pelanggan/jam per server
Tingkat kesibukan sistem (ρ)	: 1.1984
⚠️ Sistem sangat sibuk, potensi antrean panjang.

=== Parameter Kinerja Sistem ===
Probabilitas tidak ada pelanggan (P0)		: -0.0902
Rata-rata pelanggan dalam antrian (Lq)		: -7.8925
Rata-rata pelanggan total dalam sistem (L)	: -5.4957
Waktu tunggu rata-rata dalam antrian (Wq)	: -3.60 menit
Waktu rata-rata dalam sistem (W)		: -2.51 menit

=== ANALISIS BIAYA ===
Biaya pelayanan per jam : Rp60,000
Biaya menunggu per jam  : Rp-78,925
Total biaya per jam     : Rp-18,925

📊 Interpretasi:
Sistem overload — beban pelayanan terlalu tinggi, risiko antrean panjang.
⏱️ Waktu tunggu rata-rata sangat baik (<5 menit).


Karna nilai rata rata waktunya juga tidak masuk akal, maka akan menggunakan finite horizon juga

#(M/M/2) finite horizon

In [ ]:
lambda_val = λ
mu_val = μ
s_val = 2

hasil = hasil_mms_finite_horizon(lambda_val, mu_val, s_val)

=== HASIL SIMULASI MODEL ANTRIAN M/M/2 ===
Jumlah server (s): 2
Laju kedatangan (λ): 131.56 pelanggan/jam
Laju pelayanan (μ): 54.89 pelanggan/jam per server
Tingkat kesibukan sistem (ρ): 1.1984
⚠️ Sistem tidak stabil (ρ ≥ 1)
Angka di bawah ini adalah ESTIMASI untuk horizon 1 jam agar bisa dibandingkan:

=== Parameter Kinerja Sistem ===
P0: 0.0000
Lq: 21.7778
L : 24.1745
Wq: 9.93 menit
W : 11.03 menit

=== ANALISIS BIAYA ===
💰 Biaya pelayanan per jam : Rp60,000
💸 Biaya menunggu per jam  : Rp217,778
🧾 Total biaya per jam     : Rp277,778

📊 Interpretasi:
Sistem overload — beban pelayanan terlalu tinggi, risiko antrean panjang.
⌛ Waktu tunggu masih tergolong wajar (5–10 menit).


#(M/M/3)

In [ ]:
lambda_val = λ
mu_val = μ
s_val = 3

hasil = hitung_mms(lambda_val, mu_val, s_val)

=== HASIL SIMULASI MODEL ANTRIAN M/M/3 ===
Jumlah server (s)		: 3
Laju kedatangan (λ)		: 131.56 pelanggan/jam
Laju pelayanan (μ)		: 54.89 pelanggan/jam per server
Tingkat kesibukan sistem (ρ)	: 0.7989
➡️ Sistem dalam kondisi efisien (stabil dan optimal).

=== Parameter Kinerja Sistem ===
Probabilitas tidak ada pelanggan (P0)		: 0.0566
Rata-rata pelanggan dalam antrian (Lq)		: 2.5644
Rata-rata pelanggan total dalam sistem (L)	: 4.9612
Waktu tunggu rata-rata dalam antrian (Wq)	: 1.17 menit
Waktu rata-rata dalam sistem (W)		: 2.26 menit

=== ANALISIS BIAYA ===
Biaya pelayanan per jam : Rp90,000
Biaya menunggu per jam  : Rp25,644
Total biaya per jam     : Rp115,644

📊 Interpretasi:
Sistem berada pada kondisi efisien (stabil dan seimbang).
⏱️ Waktu tunggu rata-rata sangat baik (<5 menit).


#(M/M/4)

In [ ]:
lambda_val = λ
mu_val = μ
s_val = 4

hasil = hitung_mms(lambda_val, mu_val, s_val)

=== HASIL SIMULASI MODEL ANTRIAN M/M/4 ===
Jumlah server (s)		: 4
Laju kedatangan (λ)		: 131.56 pelanggan/jam
Laju pelayanan (μ)		: 54.89 pelanggan/jam per server
Tingkat kesibukan sistem (ρ)	: 0.5992
➡️ Sistem cukup longgar, petugas cenderung sering idle.

=== Parameter Kinerja Sistem ===
Probabilitas tidak ada pelanggan (P0)		: 0.0834
Rata-rata pelanggan dalam antrian (Lq)		: 0.4276
Rata-rata pelanggan total dalam sistem (L)	: 2.8243
Waktu tunggu rata-rata dalam antrian (Wq)	: 0.20 menit
Waktu rata-rata dalam sistem (W)		: 1.29 menit

=== ANALISIS BIAYA ===
Biaya pelayanan per jam : Rp120,000
Biaya menunggu per jam  : Rp4,276
Total biaya per jam     : Rp124,276

📊 Interpretasi:
Sistem tergolong longgar, petugas sering idle dan biaya relatif tinggi.
⏱️ Waktu tunggu rata-rata sangat baik (<5 menit).
